In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# Load the processed data and mappings
df_features = pd.read_parquet('../data/processed/processed_features.parquet')
track_mapping = joblib.load('../saved_models/preprocessors/track_mapping.pkl')
trackid_to_index = joblib.load('../saved_models/preprocessors/trackid_to_index.pkl')

track_ids = df_features['track_id'].values
X = df_features.drop(columns=['track_id']).values 

print(f"Data loaded. Shape: {X.shape}")

Data loaded. Shape: (89741, 33)


In [2]:
print("=== Building Markov Transition Matrix ===")

# We look at the top 20 most similar songs for each track to define our "states"
K = 20 
nn_model = NearestNeighbors(n_neighbors=K + 1, metric='cosine', algorithm='brute', n_jobs=-1)
nn_model.fit(X)

# Get distances and indices for ALL songs
distances, indices = nn_model.kneighbors(X)

# Convert Cosine Distance to Cosine Similarity (Probability)
# Cosine Distance = 1 - Similarity. So Similarity = 1 - Distance.
similarities = 1 - distances

# Remove the first column (the song itself, distance = 0)
similarities = similarities[:, 1:]
indices = indices[:, 1:]

# Normalize each row so the probabilities sum to 1 (The Markov Property)
row_sums = similarities.sum(axis=1, keepdims=True)
transition_probs = similarities / row_sums

print(f"Transition matrix built. Shape: {transition_probs.shape}")
print("Each row sums to:", transition_probs[0].sum()) # Should be 1.0

=== Building Markov Transition Matrix ===


e:\Music-Recommendation-System\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\Music-Recommendation-System\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\Music-Recommendation-System\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\Music-Recommendation-System\venv\Lib\site-packages\sklearn\utils

Transition matrix built. Shape: (89741, 20)
Each row sums to: 0.9999999999999999


In [3]:
def get_markov_recommendations(track_id, n_recommendations=5):
    idx = trackid_to_index[track_id]
    
    # 1. Get the immediate neighbors (1-step)
    step1_indices = indices[idx]
    step1_probs = transition_probs[idx]
    
    # Dictionary to accumulate 2-step probabilities
    two_step_probs = {}
    
    # 2. Calculate 2-step transitions (A -> B -> C)
    for i in range(len(step1_indices)):
        b_idx = step1_indices[i]
        prob_a_to_b = step1_probs[i]
        
        # Get neighbors of B
        step2_indices = indices[b_idx]
        step2_probs = transition_probs[b_idx]
        
        # Accumulate probability: P(A->C) += P(A->B) * P(B->C)
        for j in range(len(step2_indices)):
            c_idx = step2_indices[j]
            prob_b_to_c = step2_probs[j]
            
            # Skip if C is the original song A
            if c_idx == idx:
                continue
                
            combined_prob = prob_a_to_b * prob_b_to_c
            
            if c_idx in two_step_probs:
                two_step_probs[c_idx] += combined_prob
            else:
                two_step_probs[c_idx] = combined_prob
                
    # 3. Sort by highest 2-step probability
    sorted_tracks = sorted(two_step_probs.items(), key=lambda x: x[1], reverse=True)
    
    # 4. Format the output
    recommendations = []
    for c_idx, prob in sorted_tracks[:n_recommendations]:
        rec_track_id = track_ids[c_idx]
        recommendations.append({
            'track_name': track_mapping[rec_track_id]['track_name'],
            'artists': track_mapping[rec_track_id]['artists'],
            'track_genre': track_mapping[rec_track_id]['track_genre'],
            'probability': prob
        })
        
    return recommendations

In [4]:
import os
os.makedirs('../saved_models', exist_ok=True)

# Save the transition arrays
np.save('../saved_models/markov_indices.npy', indices)
np.save('../saved_models/markov_probs.npy', transition_probs)

print("Markov Chain model saved successfully to '../saved_models/'")

Markov Chain model saved successfully to '../saved_models/'


In [5]:
# Pick a test track
test_track_id = track_ids[100] 
print(f"\n--- Testing 2-Step Markov Chain ---")
print(f"Input Track: {track_mapping[test_track_id]['track_name']} by {track_mapping[test_track_id]['artists']} ({track_mapping[test_track_id]['track_genre']})\n")

recs = get_markov_recommendations(test_track_id, n_recommendations=5)

print("Recommended 'Next' Songs (2-Step Journey):")
for i, rec in enumerate(recs, 1):
    print(f"{i}. {rec['track_name']} by {rec['artists']} [{rec['track_genre']}] (Prob: {rec['probability']:.4f})")


--- Testing 2-Step Markov Chain ---
Input Track: Rain by Motohiro Hata (acoustic)

Recommended 'Next' Songs (2-Step Journey):
1. Change the World - 2015 Remaster by Eric Clapton [blues] (Prob: 0.0375)
2. mr. nice guy by KAYDEN [chill] (Prob: 0.0300)
3. Gloria (feat. Van Morrison) - Stereo Version by Them;Van Morrison [british] (Prob: 0.0250)
4. Make It Wit Chu by Queens of the Stone Age [blues] (Prob: 0.0250)
5. Eight Days A Week - Remastered 2009 by The Beatles [british] (Prob: 0.0225)
